# Handoff Sprint 2 của Hùng — chạy thử UrbanEV Development trên Colab

Notebook gọi các CLI tại commit đã ghim; không chứa logic mô hình. Đây là **phần chuẩn bị chưa chạy**. Lượt đầu chỉ chạy thử Source/Development với giới hạn trên bộ dữ liệu 7 ngày, đi qua một ranh giới block replay 14 ngày. Lượt này không thay thế huấn luyện Source đầy đủ, chọn tham số, bằng chứng hiệu quả mô hình hay đánh giá Final. Runner chỉ dùng CPU. Ghi lại mọi thay đổi cấu hình và lưu đầu ra lớn vào thư mục Drive mới.

Điều kiện đầu vào: đặt gói đã công bố `inputs-urbanev-SZH-v1.zip` trên Drive, điền hai đường dẫn bên dưới và dùng runtime Colab mới. Notebook kiểm tra gói theo release catalog cố định trước khi giải nén. ID Development `1001` có 15 origin được tạo trong lịch 7 ngày đã xác minh cục bộ; đây là đối tượng thử kỹ thuật, không được chọn theo hiệu quả mô hình.

Để thử riêng nhóm `source_checkpoint_only`, chạy lại trên runtime mới với `ENTITY_ID = "1025"` và `RUN_ID` mới. Theo danh mục Development 7 ngày đã xác minh, ID này có 0 mẫu target ban đầu và 15 origin được tạo; `target-smoke` cần ghi 0 bước thích nghi. Giữ kết quả nhóm này tách khỏi ID `1001`. Biến thể này chưa chạy.


In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

CODE_SHA = "85e45606789eaff7e2e929eb7a7ee353a4ee49f9"
ARCHIVE_ON_DRIVE = Path("/content/drive/MyDrive/CHANGE_ME/inputs-urbanev-SZH-v1.zip")
DRIVE_RUN_PARENT = Path("/content/drive/MyDrive/CHANGE_ME/hung-sprint2-runs")
RUN_ID = "hung-urbanev-7d-smoke-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SEED = 7
ENTITY_ID = "1001"
SMOKE = {
    "source_steps": 2,
    "target_steps": 2,
    "batch_size": 2,
    "learning_rate": 0.001,
    "retention_weight": 0.1,
    "epsilon": 1e-6,
    "max_origins": 15,
    "level_one": 0.1,
    "level_two": 0.2,
    "initial_level": 0.0,
    "block_days": 14,
    "min_complete_days": 5,
    "delta": 0.0,
    "alpha": 0.05,
    "bootstrap_segment_days": 2,
    "bootstrap_resamples": 100,
    "challenger_rule": "bonferroni_two",
}
assert "CHANGE_ME" not in str(ARCHIVE_ON_DRIVE) + str(DRIVE_RUN_PARENT)
assert ARCHIVE_ON_DRIVE.is_file()
print("ma_luot_chay", RUN_ID)

In [ ]:
repo = Path("/content/hung-sprint2-code")
if repo.exists():
    raise FileExistsError(repo)
subprocess.run(
    ["git", "clone", "https://github.com/HungLD9505/adaptive-retention-ev.git", str(repo)],
    check=True,
)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", CODE_SHA], check=True)
actual_sha = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()
assert actual_sha == CODE_SHA
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements-dev.txt")], check=True
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--index-url",
        "https://download.pytorch.org/whl/cpu",
        "-r",
        str(repo / "requirements-model.txt"),
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
print("commit_code", actual_sha)


def run_cli(*args):
    command = [sys.executable, "-m", "experiments.hung_sprint2.cli", *map(str, args)]
    print("lenh_chay:", " ".join(command))
    subprocess.run(command, cwd=repo, check=True)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_drive_run(folder):
    folder = Path(folder)
    marker = folder / "handoff_complete.json"
    if not marker.is_file():
        raise ValueError(f"Drive handoff is incomplete: {folder}")
    recorded = json.loads(marker.read_text(encoding="utf-8"))
    if (
        recorded.get("schema") != "hung-sprint2-drive-handoff-v1"
        or recorded.get("code_sha") != CODE_SHA
    ):
        raise ValueError("Unexpected Drive handoff schema/code")
    actual = {
        p.relative_to(folder).as_posix() for p in folder.rglob("*") if p.is_file() and p != marker
    }
    if actual != set(recorded["files"]):
        raise ValueError("Drive handoff file set differs")
    for name, expected in recorded["files"].items():
        if sha256_file(folder / name) != expected:
            raise ValueError(f"Drive handoff hash mismatch: {name}")
    return recorded


def publish_run(local_folder, drive_folder):
    local_folder = Path(local_folder)
    drive_folder = Path(drive_folder)
    drive_folder.parent.mkdir(parents=True, exist_ok=True)
    if drive_folder.exists():
        raise FileExistsError(drive_folder)
    shutil.copytree(local_folder, drive_folder)
    files = {}
    for original in local_folder.rglob("*"):
        if original.is_file():
            name = original.relative_to(local_folder).as_posix()
            digest = sha256_file(original)
            if digest != sha256_file(drive_folder / name):
                raise ValueError(f"Drive copy mismatch: {name}")
            files[name] = digest
    marker = {
        "schema": "hung-sprint2-drive-handoff-v1",
        "code_sha": CODE_SHA,
        "files": files,
    }
    with (drive_folder / "handoff_complete.json").open("x", encoding="utf-8") as stream:
        json.dump(marker, stream, indent=2, sort_keys=True)
        stream.write("\n")
    verify_drive_run(drive_folder)
    return drive_folder

In [ ]:
archive = Path("/content/inputs-urbanev-SZH-v1.zip")
input_root = Path("/content/hung-sprint2-data")
if archive.exists() or input_root.exists():
    raise FileExistsError("Use a fresh archive/input destination")
shutil.copyfile(ARCHIVE_ON_DRIVE, archive)
subprocess.run(
    [
        sys.executable,
        "-m",
        "experiments.hung_sprint2.cli",
        "verify-input-zip",
        "--archive",
        str(archive),
        "--package",
        archive.name,
    ],
    cwd=repo,
    check=True,
)
with zipfile.ZipFile(archive) as package:
    names = package.namelist()
    prefix = "data/inputs/issue18-input-handoff-v1-20261019/urbanev-SZH-"
    if len(names) != len(set(names)) or any(
        not name.startswith(prefix) or Path(name).is_absolute() or ".." in Path(name).parts
        for name in names
    ):
        raise ValueError("Unexpected archive entry")
    input_root.mkdir()
    package.extractall(input_root)
run = input_root / "data/inputs/issue18-input-handoff-v1-20261019/urbanev-SZH-7d"
manifest_sha = hashlib.sha256((run / "manifest.json").read_bytes()).hexdigest()
print("bo_dau_vao", run.name, "sha256_manifest", manifest_sha)

In [ ]:
run_root = Path("/content/hung-sprint2-runs") / RUN_ID
if run_root.exists():
    raise FileExistsError(run_root)
run_root.mkdir(parents=True)


source_out = run_root / "source"
target_out = run_root / "target"
replay_out = run_root / "replay-15"
run_cli(
    "source-smoke",
    "--input-run",
    run,
    "--manifest-sha256",
    manifest_sha,
    "--output",
    source_out,
    "--steps",
    SMOKE["source_steps"],
    "--batch-size",
    SMOKE["batch_size"],
    "--seed",
    SEED,
    "--learning-rate",
    SMOKE["learning_rate"],
    "--calendar-hidden",
    32,
)
run_cli(
    "target-smoke",
    "--input-run",
    run,
    "--manifest-sha256",
    manifest_sha,
    "--source-input-run",
    run,
    "--source-manifest-sha256",
    manifest_sha,
    "--source-checkpoint",
    source_out / "source_checkpoint.pt",
    "--entity-id",
    ENTITY_ID,
    "--output",
    target_out,
    "--steps",
    SMOKE["target_steps"],
    "--batch-size",
    SMOKE["batch_size"],
    "--seed",
    SEED,
    "--learning-rate",
    SMOKE["learning_rate"],
    "--retention-weight",
    SMOKE["retention_weight"],
    "--epsilon",
    SMOKE["epsilon"],
)
replay_args = [
    "--input-run",
    run,
    "--manifest-sha256",
    manifest_sha,
    "--source-input-run",
    run,
    "--source-manifest-sha256",
    manifest_sha,
    "--source-checkpoint",
    source_out / "source_checkpoint.pt",
    "--target-checkpoint",
    target_out / "target_checkpoint.pt",
    "--entity-id",
    ENTITY_ID,
    "--output",
    replay_out,
    "--max-origins",
    SMOKE["max_origins"],
    "--block-days",
    SMOKE["block_days"],
    "--min-complete-days",
    SMOKE["min_complete_days"],
    "--bootstrap-segment-days",
    SMOKE["bootstrap_segment_days"],
    "--bootstrap-resamples",
    SMOKE["bootstrap_resamples"],
    "--seed",
    SEED,
    "--level-one",
    SMOKE["level_one"],
    "--level-two",
    SMOKE["level_two"],
    "--initial-level",
    SMOKE["initial_level"],
    "--delta",
    SMOKE["delta"],
    "--alpha",
    SMOKE["alpha"],
    "--epsilon",
    SMOKE["epsilon"],
    "--challenger-rule",
    SMOKE["challenger_rule"],
]
run_cli("replay-development", *replay_args)
summary = json.loads((replay_out / "run_manifest.json").read_text())
assert summary["runner_cursor"] == SMOKE["max_origins"]
print("do_phu", summary["coverage"], "sai_so_kwh", summary["metrics_kwh"]["live"])
print("bo_nho_trang_thai_tensor", summary["state_tensor_memory"])
for label, folder in (("source", source_out), ("target", target_out), ("replay", replay_out)):
    record = json.loads((folder / "run_manifest.json").read_text())
    print(
        label,
        "thoi_gian_giay",
        record["duration_seconds"],
        "bo_nho_tien_trinh",
        record["memory_usage"],
    )

In [ ]:
# Chỉ bàn giao sau khi so hash các file đã sao chép; thư mục bị gián đoạn không có marker hoàn tất.
drive_run = publish_run(run_root, DRIVE_RUN_PARENT / RUN_ID)
print("thu_muc_drive", drive_run)
print("sha256_checkpoint_runner", summary["final_runner_checkpoint_sha256"])

## Tiếp tục từ checkpoint trên runtime mới

Chạy lại các ô thiết lập và giải nén có xác minh. Đặt `PREVIOUS_RUN_ID` bằng run ID đã bàn giao trên Drive, rồi chạy ô bên dưới. Ô này đọc checkpoint Source/target và runner cũ từ Drive; `replay-development` kiểm tra nguồn gốc, cấu hình và commit code; sau đó tạo đầu ra cục bộ **mới** và sao chép sang thư mục Drive riêng. Các dự báo cũ không được phát lại. Cấu hình smoke phía trên chỉ để xác minh kỹ thuật trên Development, chưa phải lựa chọn nghiên cứu đã khóa.


In [ ]:
PREVIOUS_RUN_ID = "CHANGE_ME_PREVIOUS_RUN_ID"
assert PREVIOUS_RUN_ID != "CHANGE_ME_PREVIOUS_RUN_ID"
previous = DRIVE_RUN_PARENT / PREVIOUS_RUN_ID
verify_drive_run(previous)
assert (previous / "replay-15/runner_checkpoint.pt").is_file()
resume_out = Path("/content/hung-sprint2-resume") / (PREVIOUS_RUN_ID + "-to-29")
if resume_out.exists():
    raise FileExistsError(resume_out)
resume_args = [
    "--input-run",
    run,
    "--manifest-sha256",
    manifest_sha,
    "--source-input-run",
    run,
    "--source-manifest-sha256",
    manifest_sha,
    "--source-checkpoint",
    previous / "source/source_checkpoint.pt",
    "--target-checkpoint",
    previous / "target/target_checkpoint.pt",
    "--entity-id",
    ENTITY_ID,
    "--output",
    resume_out,
    "--max-origins",
    29,
    "--block-days",
    SMOKE["block_days"],
    "--min-complete-days",
    SMOKE["min_complete_days"],
    "--bootstrap-segment-days",
    SMOKE["bootstrap_segment_days"],
    "--bootstrap-resamples",
    SMOKE["bootstrap_resamples"],
    "--seed",
    SEED,
    "--level-one",
    SMOKE["level_one"],
    "--level-two",
    SMOKE["level_two"],
    "--initial-level",
    SMOKE["initial_level"],
    "--delta",
    SMOKE["delta"],
    "--alpha",
    SMOKE["alpha"],
    "--epsilon",
    SMOKE["epsilon"],
    "--challenger-rule",
    SMOKE["challenger_rule"],
]
run_cli(
    "replay-development",
    *resume_args,
    "--resume-checkpoint",
    previous / "replay-15/runner_checkpoint.pt",
)
new_drive_run = publish_run(resume_out, DRIVE_RUN_PARENT / (PREVIOUS_RUN_ID + "-resume-29"))
print("thu_muc_drive_sau_resume", new_drive_run)